In [8]:
import sys
import os

# Add the project root to the Python path to allow for module imports
project_root = os.path.abspath(os.path.join(os.getcwd(), '../droid_slam'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


In [ ]:
import torch
import onnx
import netron
import collections
from droid_slam.droid_net import DroidNet
from droid_slam.droid_args import DroidArgs

# DROID is not a standard PyTorch model, so we need its definition.
# Assuming the DROID model class is defined in a 'droid_net.py' file
# as is common in DROID-SLAM implementations.

# --- 1. Load the PyTorch Model ---
pth_model_path = '../droid.pth'
onnx_model_path = '../droid.onnx'

# Instantiate the model
# Adjust arguments if your DroidNet constructor requires them

model = DroidNet()

# Load the weights from the .pth file
# The original DROID weights are often saved with a 'model' key.
state_dict = torch.load(pth_model_path)

# The state_dict might be nested or have a 'module.' prefix from DataParallel
# We create a new state_dict to handle these cases
new_state_dict = collections.OrderedDict()
for k, v in state_dict.items():
    name = k[7:] if k.startswith('module.') else k
    new_state_dict[name] = v

model.load_state_dict(new_state_dict)
model.eval()

# --- 2. Convert to ONNX ---
# DROID typically takes multiple inputs (images, intrinsics, etc.)
# We need to create dummy inputs with the correct shape and type.
# These shapes are based on common usage for DROID-SLAM.
# Please adjust these shapes if your use case is different.
#  b, n, c1, h1, w1 = x.shape
dummy_images = torch.randn(1, 2, 3, 384, 512) # (t, B, C, H, W)
dummy_intrinsics = torch.randn(1, 1, 8)       # (t, B, 8)

# Set the model to evaluation mode
model.eval()

print("Starting ONNX export...")
torch.onnx.export(model,
                  (dummy_images, dummy_intrinsics), # model input
                  onnx_model_path,
                  export_params=True,
                  opset_version=12,
                  do_constant_folding=True,
                  input_names = ['images', 'intrinsics'],
                  output_names = ['outputs'], # adjust if your model has more outputs
                  dynamic_axes={'images' : {0 : 'time'},
                                'intrinsics' : {0 : 'time'}})

print(f"Model has been converted to ONNX and saved at {onnx_model_path}")

# --- 3. Verify and Visualize the ONNX model ---
# Load the ONNX model
onnx_model = onnx.load(onnx_model_path)

# Check that the model is well-formed
onnx.checker.check_model(onnx_model)

print("ONNX model check passed.")
print("Starting Netron visualization server...")

# Visualize the model using Netron
# This will start a web server and open the model in your browser.
netron.start(onnx_model_path)

ModuleNotFoundError: No module named 'droid_slam.droid_args'